# Assignment 5: Fine Tuning FLAN T5 Small with LoRA on DialogSum

In [1]:
import datasets
import evaluate
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import DataLoader
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, logging

logging.set_verbosity_error()
logging.disable_progress_bar()
datasets.disable_progress_bars()
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "google/flan-t5-small"
pd.set_option("display.max_colwidth", None)

## 1. Data Preparation

Input is the dialogue wrapped in a short instruction prompt, the format FLAN T5 was trained on. Target is the summary.

In [2]:
ds = load_dataset("neil-code/dialogsum-test")
PROMPT = "Summarize the following conversation.\n\n{}\n\nSummary:"
data = ds.map(lambda x: {"input": PROMPT.format(x["dialogue"]), "target": x["summary"]})
print(ds)
for x in data["train"].select(range(2)):
    print("INPUT:\n" + x["input"] + "\n\nTARGET:\n" + x["target"] + "\n")

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})
INPUT:
Summarize the following conversation.

#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoki

In [3]:
tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(b):
    enc = tok(b["input"], max_length=512, truncation=True)
    enc["labels"] = tok(text_target=b["target"], max_length=128, truncation=True)["input_ids"]
    return enc

train = data["train"].map(tokenize, batched=True, remove_columns=data["train"].column_names)

## 2. Baseline Inference

In [4]:
@torch.no_grad()
def summarize(model, dialogues):
    model.eval()
    enc = tok([PROMPT.format(d) for d in dialogues], max_length=512, truncation=True, padding=True, return_tensors="pt").to(device)
    return tok.batch_decode(model.generate(**enc, max_new_tokens=100), skip_special_tokens=True)

base = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device)
samples = ds["test"].select([0, 3])
baseline = summarize(base, samples["dialogue"])
for d, out in zip(samples["dialogue"], baseline):
    print(d, "\n\nBASELINE:", out, "\n")
pd.DataFrame({"reference": samples["summary"], "baseline": baseline}).to_csv("../baseline_outputs.csv", index=False)

#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, please continue with the memo. Wh

## 3. Apply LoRA (r = 16, alpha = 32, dropout = 0.05)

alpha is set to 2r, so alpha / r = 2 for both ranks in Task 7. The seed is reset before each model is built.

In [5]:
def lora_model(r):
    torch.manual_seed(42)
    config = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, target_modules=["q", "v"], task_type=TaskType.SEQ_2_SEQ_LM)
    model = get_peft_model(AutoModelForSeq2SeqLM.from_pretrained(MODEL), config).to(device)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"r={r}  Total parameters: {total:,}  Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}%)")
    return model

model16 = lora_model(16)

r=16  Total parameters: 77,649,280  Trainable parameters: 688,128 (0.89%)


## 4. Fine Tuning (full train split, 1,999 dialogues, 5 epochs)

In [6]:
def finetune(model, epochs=5):
    loader = DataLoader(train, batch_size=8, shuffle=True, collate_fn=DataCollatorForSeq2Seq(tok))
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=0.001)
    model.train()
    for epoch in range(epochs):
        losses = []
        for batch in loader:
            loss = model(**batch.to(device)).loss
            loss.backward()
            opt.step()
            opt.zero_grad()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}  training loss {sum(losses) / len(losses):.4f}")
    return sum(losses) / len(losses)

loss16 = finetune(model16)
print("Final training loss:", round(loss16, 4))
model16.save_pretrained("../models/lora_r16")

epoch 1  training loss 1.5763


epoch 2  training loss 1.4662


epoch 3  training loss 1.4179


epoch 4  training loss 1.3747


epoch 5  training loss 1.3394
Final training loss: 1.3394


## 5. Post Fine Tuning Inference

In [7]:
tuned = summarize(model16, samples["dialogue"])
for ref, out in zip(samples["summary"], tuned):
    print("REFERENCE: ", ref, "\nFINE TUNED:", out, "\n")

REFERENCE:  Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore. 
FINE TUNED: Ms. Dawson wants to take a dictation for Ms. Dawson. Ms. Dawson tells Ms. Dawson that all office communications are restricted to email correspondence and official memos. Ms. Dawson tells Ms. Dawson that the use of Instant Messaging is strictly prohibited. Ms. Dawson tells Ms. Dawson that the use of Instant Messaging is strictly prohibited. 

REFERENCE:  #Person2# arrives late because of traffic jam. #Person1# persuades #Person2# to use public transportations to keep healthy and to protect the environment. 
FINE TUNED: #Person2# got stuck in traffic again. #Person1# thinks the public transport system is good and the public transport system is good. #Person2# thinks it's better for the environment and the environment. 



## 6. Output Comparison

In [8]:
pd.DataFrame({"reference": samples["summary"], "before": baseline, "after (r=16)": tuned})

,reference,before,after (r=16)
0,Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.,Is this all correct?,Ms. Dawson wants to take a dictation for Ms. Dawson. Ms. Dawson tells Ms. Dawson that all office communications are restricted to email correspondence and official memos. Ms. Dawson tells Ms. Dawson that the use of Instant Messaging is strictly prohibited. Ms. Dawson tells Ms. Dawson that the use of Instant Messaging is strictly prohibited.
1,#Person2# arrives late because of traffic jam. #Person1# persuades #Person2# to use public transportations to keep healthy and to protect the environment.,Talk to the driver.,#Person2# got stuck in traffic again. #Person1# thinks the public transport system is good and the public transport system is good. #Person2# thinks it's better for the environment and the environment.


Before fine tuning the model does not summarize: it outputs unrelated fragments ("Is this all correct?", "Talk to the driver."). After LoRA fine tuning it writes summaries in the DialogSum style that capture the main topics (the Instant Messaging ban, the traffic, public transport), using #Person1# and #Person2# tags in the traffic summary. It is still imperfect, with repetition and confused roles ("Ms. Dawson tells Ms. Dawson").

## 7. Experiment: r = 4 vs r = 16

In [9]:
model4 = lora_model(4)
loss4 = finetune(model4)
model4.save_pretrained("../models/lora_r4")

r=4  Total parameters: 77,133,184  Trainable parameters: 172,032 (0.22%)


epoch 1  training loss 1.6227


epoch 2  training loss 1.4931


epoch 3  training loss 1.4593


epoch 4  training loss 1.4328


epoch 5  training loss 1.4072


In [10]:
rouge = evaluate.load("rouge")
test = ds["test"].select(range(0, 300, 3))

def score(model):
    preds = [p for i in range(0, 100, 20) for p in summarize(model, test["dialogue"][i:i + 20])]
    return {k: round(v, 4) for k, v in rouge.compute(predictions=preds, references=test["summary"]).items()}

pd.DataFrame({"baseline": score(base), "r=4": score(model4), "r=16": score(model16)}).T.assign(
    final_train_loss=[None, round(loss4, 4), round(loss16, 4)]
)

,rouge1,rouge2,rougeL,rougeLsum,final_train_loss
baseline,0.1010,0.0228,0.0911,0.0915,NaN
r=4,0.3642,0.1305,0.2993,0.2990,1.4072
r=16,0.3733,0.1280,0.3010,0.3007,1.3394


In [11]:
pd.DataFrame({"reference": samples["summary"], "r=4": summarize(model4, samples["dialogue"]), "r=16": tuned})

,reference,r=4,r=16
0,Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.,#Person1# asks Ms. Dawson to take a dictation for Ms. Dawson. Dawson asks #Person1# to take a dictation for Ms. Dawson. Dawson asks #Person1# to take a dictation for Ms. Dawson. Dawson asks #Person1# to take a dictation for Ms. Dawson.,Ms. Dawson wants to take a dictation for Ms. Dawson. Ms. Dawson tells Ms. Dawson that all office communications are restricted to email correspondence and official memos. Ms. Dawson tells Ms. Dawson that the use of Instant Messaging is strictly prohibited. Ms. Dawson tells Ms. Dawson that the use of Instant Messaging is strictly prohibited.
1,#Person2# arrives late because of traffic jam. #Person1# persuades #Person2# to use public transportations to keep healthy and to protect the environment.,#Person2# got stuck in traffic again and #Person1# thinks it's better if #Person2# started taking public transport system to work. #Person2# thinks it's better for the environment and #Person2#'ll quit driving to work.,#Person2# got stuck in traffic again. #Person1# thinks the public transport system is good and the public transport system is good. #Person2# thinks it's better for the environment and the environment.


Both ranks score almost the same on 100 test dialogues (ROUGE L 0.299 for r = 4 vs 0.301 for r = 16, while ROUGE 2 slightly favors r = 4), so with one seed there is no measurable difference, even though r = 16 trains 4 times more parameters and reaches a lower training loss (1.339 vs 1.407). On the two samples the results are mixed: r = 4 gives the better traffic summary (it captures the switch to public transport without repeating), but on the memo dialogue it loops on "take a dictation" and misses the Instant Messaging ban that r = 16 captures. The smaller rank is a reasonable choice for cost, at a quarter of the trainable parameters, but the results do not show that it is better.